In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here: Read the dataset Q1_data.csv using read_csv()

food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

In [ ]:
# Task 2: Write your code here:Inspect the first few rows using head()
print(f"Dataset shape: {df_food.shape}")
df_food.head()


In [ ]:
# Task 3: Write your code here: Display dataset information using info()
df_food.info()

In [ ]:
# Task 4: Write your code here: Show statistical description using describe()
df_food.describe()

In [ ]:
# Task 5: Write your code here: Plot the target distribution (delivery_time)

plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('delivery_time Distribution')
plt.xlabel('delivery_time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here: Drop the 'Order_ID' column from the data

df_food = df_food.drop("Order_ID", axis=1)
df_food.head()

In [ ]:
# Task 2: Write your code here: Handle missing values appropriately (Hint: I guess you want to have a closer look at the columns with missing values :) )
print(df_food.isnull().sum())
df_food = df_food.dropna(subset=['Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs', 'Delivery_Time'])
print(df_food.isnull().sum())


In [ ]:
# Task 3: Write your code here: Check and remove duplicates if any exist
# 4. Do we have duplicate samples?
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_food)


In [ ]:
# Task 4: Write your code here: Encode categorical variables if needed (Bonus if used One Hot Encoding)
from sklearn.preprocessing import LabelEncoder

categorical_cols = df_food.select_dtypes(include=["object"]).columns

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_food[col] = le.fit_transform(df_food[col])
  label_encoders[col] = le

df_food.head()

In [ ]:
# Task 5: Write your code here: Apply feature scaling for all features (Use StandardScaler)
from sklearn.preprocessing import StandardScaler

numerical_cols = df_food.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")

scaler = StandardScaler()
df_food[numerical_cols] = scaler.fit_transform(df_food[numerical_cols])
df_food.head()


In [ ]:
# Task 6: Write your code here:Check for target imbalance and state if it is imbalanced or not (keep this cell empty if not needed)

# no needed

In [ ]:
# Task 1: Write your code here: Split the dataset into features (X) and target (y)
X = df_food.drop("Delivery_Time", axis=1)
y = df_food['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:

model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)

# K-Fold Cross Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    # Train model
  model.fit(X_train, y_train)

    # Predict
  y_pred = model.predict(X_test)

    # Evaluation metrics
  mae_scores.append(mean_absolute_error(y_test, y_pred))


In [ ]:
# Task 1: Write your code here:
feature_cols = X.columns
feature_importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
df_ypred = pd.DataFrame(y_pred)
df_ypred

In [ ]:
# Task 2: Write your code here:

# Condition distribution
condition_counts = df_ypred[0].value_counts()
plt.figure(figsize=(10, 5))
plt.bar(condition_counts.index, condition_counts.values, color='coral')
plt.title('Y_pred Distribution')
plt.xlabel('y_PRED')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Task Bonus: Write your code here: